In [1]:
# CELL 1

!pip install -q langchain langgraph langchain-openai langchain-community duckduckgo-search ddgs


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
# CELL 2

import os
import requests
from typing import Annotated, TypedDict

from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain_community.tools import DuckDuckGoSearchRun

from langgraph.graph import StateGraph, START
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition


# GROQ API KEY (your key starts with gsk_, which is a Groq key, not an xAI key)
os.environ["GROQ_API_KEY"] = "gsk_...key"

# OPTIONAL
os.environ["ALPHA_VANTAGE_API_KEY"] = "key"


/tmp/ipykernel_5000/2530378933.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


In [ ]:
# SEARCH TOOL
search_tool = DuckDuckGoSearchRun()

In [ ]:
# CALCULATOR TOOL
@tool
def calculator(num1: float, num2: float, operation: str) -> float:
    """
    Performs mathematical operations.
    """

    if operation == "add":
        return num1 + num2

    elif operation == "subtract":
        return num1 - num2

    elif operation == "multiply":
        return num1 * num2

    elif operation == "divide":

        if num2 == 0:
            return "Error: Division by zero"

        return num1 / num2

    return "Invalid operation"

In [ ]:
# STOCK TOOL
@tool
def get_stock_price(ticker: str) -> dict:
    """
    Fetches live stock price data.
    """

    api_key = os.getenv("ALPHA_VANTAGE_API_KEY")

    url = (
        f"https://www.alphavantage.co/query?"
        f"function=GLOBAL_QUOTE&symbol={ticker}&apikey={api_key}"
    )

    response = requests.get(url)

    return response.json()

In [ ]:
# TOOL LIST
tools = [
    search_tool,
    calculator,
    get_stock_price
]

In [ ]:
# GROQ MODEL (free tier)
llm = ChatOpenAI(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1"
)


# BIND TOOLS
llm_with_tools = llm.bind_tools(tools)

In [ ]:
# STATE
class State(TypedDict):
    messages: Annotated[list, add_messages]


# CHAT NODE
def chat_node(state: State):

    response = llm_with_tools.invoke(
        state["messages"]
    )

    return {
        "messages": [response]
    }

In [ ]:
# TOOL NODE
tool_node = ToolNode(tools)


# GRAPH
workflow = StateGraph(State)

workflow.add_node("chat_node", chat_node)
workflow.add_node("tools", tool_node)

workflow.add_edge(START, "chat_node")

workflow.add_conditional_edges(
    "chat_node",
    tools_condition
)

workflow.add_edge("tools", "chat_node")

chatbot = workflow.compile()

In [10]:
# RUN FUNCTION
#if we use invoke it give the output all in one time
def ask_agent(user_input):

    inputs = {
        "messages": [("user", user_input)]
    }

    print("\nAI RESPONSE:\n")

    # this will make sure the out will be displayed token by token
    for event in chatbot.stream(
        inputs,
        stream_mode="values"
    ):

        # this is for getting the output
        last_message = event["messages"][-1]

        if last_message.type == "ai":

            if last_message.content:
                print(last_message.content)


In [4]:
# CELL 4

ask_agent("What is 25 multiplied by 8?")



AI RESPONSE:

The result of 25 multiplied by 8 is 200.


In [9]:
# CELL 5

ask_agent("What is Tesla stock price right now?")



AI RESPONSE:

The current stock price of Tesla is $406.43.


In [6]:
# CELL 6

ask_agent("Who is the CEO of OpenAI?")



AI RESPONSE:

The current CEO of OpenAI is not explicitly stated in the search results provided. Let me try searching again.


The current CEO of OpenAI is Sam Altman.
